In [ ]:
import pandas as pd

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train.head()        # shows first 5 rows

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
train.info()        # shows columns, data types, missing values
train.describe()    # shows statistics like mean, min, max

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [ ]:
# Fill missing Age with the median age
train['Age'].fillna(train['Age'].median(), inplace=True)
test['Age'].fillna(test['Age'].median(), inplace=True)

# Fill missing Embarked with most common value
train['Embarked'].fillna(train['Embarked'].mode()[0], inplace=True)

# Fill missing Fare in test
test['Fare'].fillna(test['Fare'].median(), inplace=True)

# Convert Sex column: male=0, female=1
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})
test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})

# Convert Embarked: S=0, C=1, Q=2                  #embarked are port names...a categorical feature indicating the port where a passenger boarded the ship
train['Embarked'] = train['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
test['Embarked'] = test['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

/tmp/ipykernel_2986/348714705.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train['Age'].fillna(train['Age'].median(), inplace=True)
/tmp/ipykernel_2986/348714705.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)'

In [ ]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

X_train = train[features]   # input columns
y_train = train['Survived'] # target column

X_test = test[features]

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Training accuracy:", model.score(X_train, y_train))

Training accuracy: 0.9797979797979798


In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X_train, y_train, cv=5)
print("Cross-validation accuracy:", scores.mean().round(3))

Cross-validation accuracy: 0.811


In [ ]:
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})

submission.to_csv('submission.csv', index=False)

In [ ]:
# 1. Title extraction — "Mr", "Mrs", "Miss" tells a lot about survival
train['Title'] = train['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test['Title'] = test['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Simplify rare titles into one group
train['Title'] = train['Title'].replace(['Lady','Countess','Capt','Col','Don',
                                          'Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare')
test['Title'] = test['Title'].replace(['Lady','Countess','Capt','Col','Don',
                                          'Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare')

train['Title'] = train['Title'].map({'Mr':0,'Miss':1,'Mrs':2,'Master':3,'Rare':4})
test['Title'] = test['Title'].map({'Mr':0,'Miss':1,'Mrs':2,'Master':3,'Rare':4})
train['Title'].fillna(0, inplace=True)
test['Title'].fillna(0, inplace=True)

# 2. Family size — alone vs large family survival rates differ a lot
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1

# 3. Is the person alone?
train['IsAlone'] = (train['FamilySize'] == 1).astype(int)
test['IsAlone'] = (test['FamilySize'] == 1).astype(int)

/tmp/ipykernel_2986/2132717076.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train['Title'].fillna(0, inplace=True)
/tmp/ipykernel_2986/2132717076.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'd

In [ ]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch',
            'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone']

X_train = train[features]
y_train = train['Survived']
X_test = test[features]

# Retrain
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

scores = cross_val_score(model, X_train, y_train, cv=5)
print("New accuracy:", scores.mean().round(3))

New accuracy: 0.811


In [ ]:
from sklearn.model_selection import GridSearchCV

# Settings you want to try
param_grid = {
    'n_estimators': [100, 200, 300],      # how many trees
    'max_depth': [4, 6, 8, None],         # how deep each tree grows
    'min_samples_split': [2, 5, 10],      # min samples needed to split a node
    'min_samples_leaf': [1, 2, 4]         # min samples at the end of a tree
}

# This will try every combination and find the best one
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,             # same honest 5-fold testing
    scoring='accuracy',
    n_jobs=-1         # uses all your CPU cores to go faster
)

grid_search.fit(X_train, y_train)

print("Best settings:", grid_search.best_params_)
print("Best accuracy:", grid_search.best_score_.round(3))

Best settings: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 100}
Best accuracy: 0.841


In [13]:
# This is already the best version of your model
best_model = grid_search.best_estimator_

# Verify the score
scores = cross_val_score(best_model, X_train, y_train, cv=5)
print("Tuned model accuracy:", scores.mean().round(3))

Tuned model accuracy: 0.841


In [14]:
# Make predictions with your best tuned model
predictions = best_model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})

submission.to_csv('submission.csv', index=False)